# 6. Classification

Predict the 5 NewsGuard levels (logistic regression with balanced classes; ModernBERT with balanced cross-entropy). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets. ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/classification/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [1]:
TASK = "classification"
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [2]:
# Parameters
MODELS = "ModernBERT"


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-11-12 06:59:33  classification, ModernBERT, fold 1, epoch 1/5: loss 1.1958 (1008 s)


2026-11-12 07:16:16  classification, ModernBERT, fold 1, epoch 2/5: loss 0.6934 (2011 s)


2026-11-12 07:33:01  classification, ModernBERT, fold 1, epoch 3/5: loss 0.5401 (3016 s)


2026-11-12 07:49:54  classification, ModernBERT, fold 1, epoch 4/5: loss 0.4638 (4029 s)


2026-11-12 08:08:08  classification, ModernBERT, fold 1, epoch 5/5: loss 0.4375 (5122 s)


... (7 outputs left out)

2026-11-12 09:42:30  classification, ModernBERT, fold 2: done in 5571 s, GPU peak 9.7 GB


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-11-12 10:00:29  classification, ModernBERT, fold 3, epoch 1/5: loss 1.1954 (1067 s)


2026-11-12 10:18:04  classification, ModernBERT, fold 3, epoch 2/5: loss 0.6899 (2121 s)


In [ ]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())